# SPACE-GM (real package) — fast CV walkthrough

Run the genuine [`spacegm`](https://gitlab.com/enable-medicine-public/space-gm) model on the TME benchmark
with **amortized datasets**: each `spacegm.data.CellularGraphDataset` is built and processed **once** — not
rebuilt on every fold — then sliced by `train_inds` / `valid_inds`. Same patient-level cross-validation and
cohort-generalization logic as the benchmark's `cross_validate` / `cohort_split_test`, just far faster when
graph processing dominates.

Helpers live in `benchmark.models.space_gm_real_cv`:

- `cross_validate_fast(ds, task, cfg, seeds=...)` — one dataset over all CV regions, a fresh model per fold.
- `cohort_generalization_fast(ds, task, gt, cfg, seeds=...)` — two datasets (train / test cohort), sharing the
  vocabulary derived from the training cohort.

**Why it is leakage-free:** SPACE-GM featurizes each region deterministically from the explicitly-passed
vocabulary (`cell_type_mapping` / `cell_type_freq` / `biomarkers`) and fixed feature bounds — there are no
dataset-level statistics (the sanity check in section 1b confirms our featurizer reproduces
`spacegm.construct_graph_for_region` exactly). Training samples only `train_inds`
(`SubgraphSampler(selected_inds=...)`) and inference only `valid_inds`
(`collect_predict_for_all_nodes(inds=...)`), both using absolute region indices — so validation regions never
influence the fitted model even though their graphs share the dataset.

> Run this in the SPACE-GM conda env (e.g. `p3`).

## 0. Setup

In [2]:
import sys, time
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO))

import spacegm as sg
from benchmark.utils.registry import load_dataset
from benchmark.models.space_gm_real_cv import (
    SpaceGMConfig, cross_validate_fast, cohort_generalization_fast,
    derive_vocabulary, build_dataset, train_on_inds, predict_on_inds)
from benchmark.validation import summarize_folds, PRIMARY_METRIC
print('spacegm', sg.__version__)

/Users/zhenqin/miniconda3/envs/p3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


spacegm 0.1.4


## 1. Dataset + config

A small, fast configuration for the live demo. Cell-type is the only node feature by default
(`use_center/neighbor_node_features == ['cell_type']`); set `model_dir` to persist weights.

In [3]:
ds = load_dataset('hnc_wu2022')
print('tasks:', [(t, ds.get_task_config(t)['type']) for t in ds.task_ids])

cfg = SpaceGMConfig(emb_dim=512, num_iterations=1e4, batch_size=64, device='cpu',
                    eval_subsample_ratio=0.1, model_dir='./_sgm_cv_weights')
print('features -> center:', cfg.use_center_node_features, '| neighbor:', cfg.use_neighbor_node_features)

tasks: [('primary_outcome', 'binary_classification'), ('hpv_status', 'binary_classification'), ('OS', 'survival')]
features -> center: ['cell_type'] | neighbor: ['cell_type']


In [8]:
# import os, tempfile
# import numpy as np
# import pandas as pd
# import networkx as nx
# from spacegm import graph_build as gb
# from benchmark.features.space_gm_real import SpaceGMGraphBuilder

# # --- Build the SAME region two ways and compare -----------------------------
# rid = ds.get_task_metadata('primary_outcome').query("dataset == 'UPMC_HNC'")['region_id'].iloc[0]
# region = ds.load_regions([rid], normalize=False)[0]
# idx = region.coordinates.index
# mpp = float(region.microns_per_pixel)
# CUTOFF_UM = 20.0  # near-edge cutoff used by both paths

# # (A) The genuine spacegm entry point: construct_graph_for_region(graph_source='cell').
# #     It reads per-cell CSVs, so we dump this region's RAW (pixel) coords / types / expression.
# tmp = tempfile.mkdtemp(prefix='sgm_sanity_')
# coords_csv = os.path.join(tmp, 'coords.csv')
# types_csv  = os.path.join(tmp, 'types.csv')
# expr_csv   = os.path.join(tmp, 'expr.csv')
# pd.DataFrame({'CELL_ID': idx,
#               'X': region.coordinates['x'].to_numpy(),
#               'Y': region.coordinates['y'].to_numpy()}).to_csv(coords_csv, index=False)
# pd.DataFrame({'CELL_ID': idx,
#               'CELL_TYPE': region.cell_types['cell_type'].reindex(idx).to_numpy()}).to_csv(types_csv, index=False)
# expr = region.expression.reindex(idx).copy()
# expr.insert(0, 'CELL_ID', idx)
# expr.to_csv(expr_csv, index=False)

# G_real = gb.construct_graph_for_region(
#     rid,
#     cell_coords_file=coords_csv,
#     cell_types_file=types_csv,
#     cell_biomarker_expression_file=expr_csv,
#     # raw px coords -> pass physical um_per_pixel so the cutoff is evaluated in microns
#     edge_kwargs={'neighbor_edge_cutoff': CUTOFF_UM, 'um_per_pixel': mpp},
#     graph_source='cell',
# )

# # (B) Our featurizer (fit on this one region so no cell type is remapped to Unassigned).
# builder = SpaceGMGraphBuilder(near_edge_um=CUTOFF_UM).fit([region])
# G_ours = builder.transform([region]).iloc[0]['nx_graph']

# def edge_cellid_set(G):
#     """Edges as {frozenset(cell_id_a, cell_id_b)}, dropping self-loops."""
#     cid = nx.get_node_attributes(G, 'cell_id')
#     return {frozenset((cid[a], cid[b])) for a, b in G.edges() if a != b}

# def node_attr(G, key):
#     cid = nx.get_node_attributes(G, 'cell_id')
#     return {cid[n]: d.get(key) for n, d in G.nodes(data=True)}

# def edge_attr(G, key):
#     cid = nx.get_node_attributes(G, 'cell_id')
#     return {frozenset((cid[a], cid[b])): d.get(key) for a, b, d in G.edges(data=True) if a != b}

# Er, Eo = edge_cellid_set(G_real), edge_cellid_set(G_ours)
# union = Er | Eo
# print('nodes         real/ours : %d / %d' % (G_real.number_of_nodes(), G_ours.number_of_nodes()))
# print('edges         real/ours : %d / %d' % (len(Er), len(Eo)))
# print('edge topology IDENTICAL  : %s   (Jaccard %.4f)' % (Er == Eo, len(Er & Eo) / len(union)))

# # cell_type must agree cell-for-cell
# ctr, cto = node_attr(G_real, 'cell_type'), node_attr(G_ours, 'cell_type')
# ct_agree = np.mean([str(ctr[c]) == str(cto[c]) for c in ctr])
# print('cell_type agreement      : %.4f' % ct_agree)

# # edge_type (neighbor/distant) must agree on shared edges
# etr, eto = edge_attr(G_real, 'edge_type'), edge_attr(G_ours, 'edge_type')
# shared = Er & Eo
# et_agree = np.mean([etr[e] == eto[e] for e in shared])
# print('edge_type agreement      : %.4f' % et_agree)

# # --- Expected, DELIBERATE differences (not bugs) ----------------------------
# n0r, n0o = next(iter(G_real.nodes)), next(iter(G_ours.nodes))
# bm = sorted(G_ours.nodes[n0o]['biomarker_expression'])[0]
# dist_real = next(iter(edge_attr(G_real, 'distance').values()))
# dist_ours = next(iter(edge_attr(G_ours, 'distance').values()))
# bm_real = G_real.nodes[n0r]['biomarker_expression'][bm.upper()]
# bm_ours = G_ours.nodes[n0o]['biomarker_expression'][bm]
# print('\nDeliberate differences:')
# print('  voronoi_polygon  real: %s | ours: %s'
#       % (type(G_real.nodes[n0r]['voronoi_polygon']).__name__, G_ours.nodes[n0o]['voronoi_polygon']))
# print('  distance units   real: pixels  (e.g. %.2f) | ours: microns (e.g. %.2f)' % (dist_real, dist_ours))
# print('  biomarker %-8s real: raw %.3f | ours: z-scored %.3f' % (bm, bm_real, bm_ours))

## 2. Fast cross-validation (dataset processed **once**)

`cross_validate_fast` loads every CV region, featurizes it, and builds a single `CellularGraphDataset`
up front. It then loops folds, training a fresh model on each fold's `train_inds` and predicting on its
`valid_inds`. Returns one metric dict per (seed, fold) — the same shape as the benchmark's `cross_validate`.

> Keep `num_iterations` small here; the up-front featurization of all UPMC regions is the main wait.

In [4]:
from benchmark.models.space_gm_real_cv import _make_spec, derive_vocabulary, build_dataset, train_on_inds, predict_on_inds

In [ ]:
task_id = 'hpv_status'
task = task_id
metric = PRIMARY_METRIC[ds.get_task_config(task)['type']]

cfg.num_iterations = 1000
cfg.eval_subsample_ratio = 0.1

In [ ]:
vcfg = ds.validation_config
n_folds = vcfg.get("n_folds", 5)
patient_col = vcfg.get("patient_col", "patient_id")
cv_filter = vcfg.get("cv_filter")
task_cfg = ds.get_task_config(task_id)

meta = ds.get_task_metadata(task_id)

if cv_filter:
    sub = meta.query(cv_filter)
    meta = sub if len(sub) else meta

region_ids = meta["region_id"].tolist()
regions = ds.load_regions(region_ids, normalize=True)

# ---- featurize + build ONE dataset over all CV regions (processed once) ----
builder = SpaceGMGraphBuilder(near_edge_um=cfg.near_edge_um).fit(regions)
feats = builder.transform(regions)
graphs = list(feats["nx_graph"])
mapping, freq, biomarkers = derive_vocabulary(graphs)

# ---- build label file and label transform ----
tmp = tempfile.TemporaryDirectory(prefix="sgm_cv_", dir=work_root)
target_all = ds.build_target(list(feats.index), task_id)
label_transform, loss_fn, num_tasks, classes, postprocess = _make_spec(task_cfg, target_all, tmp.name)

# ---- build dataset (processed once, shared by all folds) ----
dataset = build_dataset(graphs, os.path.join(tmp.name, "ds"), cfg, mapping, freq, biomarkers)
transforms = [sg.transform.FeatureMask(
    dataset,
    use_center_node_features=cfg.use_center_node_features,
    use_neighbor_node_features=cfg.use_neighbor_node_features)]
if label_transform is not None:
    transforms.append(label_transform)
dataset.transform = transforms
ridx = {str(r): i for i, r in enumerate(dataset.region_ids)}
idxr = {i: str(r) for i, r in enumerate(dataset.region_ids)}

Building graphs for 308 regions...


In [ ]:
fold_metrics: list[dict] = []

seed = 0
folds = safe_patient_kfold(meta, n_folds, patient_col, stratify_column(task_cfg), seed)

for fold_i, (train_ids, val_ids) in enumerate(folds):
    tr_inds = [ridx[str(r)] for r in train_ids if str(r) in ridx]
    va_inds = [ridx[str(r)] for r in val_ids if str(r) in ridx]

    model, device = train_on_inds(
        dataset, cfg, seed, num_tasks, loss_fn, tr_inds, mapping,
        model_dir=model_dir, tag=f"{task_id}_fold{fold_i}")
    pred_map = predict_on_inds(model, dataset, cfg, va_inds, device)

    y_va = target_all.loc[[idxr[i] for i in va_inds]]
    y_pred = postprocess(pred_map, {i: idxr[i] for i in va_inds})
    metrics = score_predictions(task_cfg, y_va, y_pred, classes)

    metrics.update({"seed": seed, "fold": fold_i,
                    "n_train": len(tr_inds), "n_val": len(va_inds)})
    fold_metrics.append(metrics)

pd.DataFrame(fold_metrics)['auc_roc'].mean()

In [4]:
import os
os.makedirs("UPMC_hpv_status_cv/data", exist_ok=True)
os.makedirs("UPMC_hpv_status_cv/models", exist_ok=True)

In [5]:
task = 'hpv_status'
metric = PRIMARY_METRIC[ds.get_task_config(task)['type']]

t0 = time.time()
folds = cross_validate_fast(
    ds, task, cfg, seeds=[0, 1, 2],
    model_dir=f"UPMC_{task}_cv/models",
    work_root=f"UPMC_{task}_cv/data")

mean, sd = summarize_folds(folds, metric)
print(f'{task}: {metric} = {mean:.3f} +/- {sd:.3f}  over {len(folds)} folds  ({time.time()-t0:.0f}s)')
for m in folds:
    print('  fold', m['fold'], '| n_train', m['n_train'], '| n_val', m['n_val'], '|',
          metric, round(m[metric], 3))

Building graphs for 308 regions...


Processing...
Done!


Finished iterations 1000, graph loss 0.59
Finished iterations 2000, graph loss 0.59
Finished iterations 3000, graph loss 0.59
Finished iterations 4000, graph loss 0.58
Finished iterations 5000, graph loss 0.58
Finished iterations 6000, graph loss 0.57
Finished iterations 7000, graph loss 0.57
Finished iterations 8000, graph loss 0.56
Finished iterations 9000, graph loss 0.56
Finished iterations 10000, graph loss 0.55
Finished iterations 1000, graph loss 0.58
Finished iterations 2000, graph loss 0.56
Finished iterations 3000, graph loss 0.55
Finished iterations 4000, graph loss 0.56
Finished iterations 5000, graph loss 0.54
Finished iterations 6000, graph loss 0.56
Finished iterations 7000, graph loss 0.54
Finished iterations 8000, graph loss 0.54
Finished iterations 9000, graph loss 0.54
Finished iterations 10000, graph loss 0.54
Finished iterations 1000, graph loss 0.59
Finished iterations 2000, graph loss 0.59
Finished iterations 3000, graph loss 0.58
Finished iterations 4000, graph 

: 

## 3. Cohort generalization (two datasets)

The generalization test trains on UPMC and tests on DFCI, using the cohort-uniform cell types. Because the
two cohorts have different cell-type vocabularies, **two** datasets are built — but both share the mapping
derived from the *training* cohort, so the model's cell-type embeddings line up.

In [ ]:
gt = ds.validation_config['generalization_tests'][0]
print('gen test:', gt['name'], '| train', gt['train'], '-> test', gt['test'],
      '| cell_type_col', gt.get('cell_type_col'))

gtask = gt.get('tasks', ds.task_ids)[0]
gmetric = PRIMARY_METRIC[ds.get_task_config(gtask)['type']]
res = cohort_generalization_fast(ds, gtask, gt, cfg, seeds=[0])
mean, sd = summarize_folds(res, gmetric)
print(f"{gtask} [{gt['name']}]: {gmetric} = {mean:.3f} +/- {sd:.3f}")
for m in res:
    print('  seed', m['seed'], '| n_train', m['n_train'], '| n_test', m['n_test'])

## 5. Running the full benchmark

`scripts/run_space_gm_real_cv.py` sweeps all datasets / tasks / schemes with the amortized datasets and
writes `results/space_gm_real_cv_benchmark.csv`.

```bash
# full run (GPU, manuscript-ish budget), saving every trained model
python scripts/run_space_gm_real_cv.py \
    --device cuda --num-iterations 50000 --emb-dim 512 \
    --model-dir results/space_gm_real_weights --work-root /fast/scratch

# quick smoke test
python scripts/run_space_gm_real_cv.py --datasets hnc_wu2022 --seeds 0 \
    --num-iterations 300 --device cpu
```

`--work-root` controls where the temporary datasets are processed (point it at fast local disk for big
cohorts); `--model-dir` saves weights, one subfolder per (task, fold, seed).